In [ ]:
import pandas as pd
import geopandas as gpd
import requests as re
from io import BytesIO
from os import path, makedirs

from core.downloads.orcamento import load_orcamento, load_orcamento_r

## Orçamento previsto/liquidado da função habitação, programa 3008 e projetos atividades 1701 e 1702

Vamos começar coletando os dados orçamentários do site de execução orçamentária da Secretaria da Fazenda.

In [ ]:
ANOS = [2024, 2025]

df_orcamento = pd.concat(
    [load_orcamento(ano).assign(ANO=str(ano)) for ano in ANOS],
    ignore_index=True
)
df_orcamento

In [ ]:
for col in [col for col in df_orcamento.columns if 'Vl' in col]:
    df_orcamento.loc[:, col] = df_orcamento[col].str.replace(',', '.').astype(float)
df_orcamento.loc[:, 'DataExtracao'] = pd.to_datetime(df_orcamento['DataExtracao'], format='%d/%m/%Y')
df_orcamento

In [ ]:
filtro_orcamento = (
    (df_orcamento['Cd_Funcao']=='16') |
    (df_orcamento['Cd_Programa']=='3008') |
    (df_orcamento['ProjetoAtividade'].isin(['1702', '1703']))
)

df_orcamento = df_orcamento.loc[filtro_orcamento]
df_orcamento

## Orçamento regionalizado no Programa 3002

O orçamento é regionalizado apenas na liquidação, então não é possível obter o orçamento previsto por subprefeitura, mas é possível obter o orçamento liquidado.

In [ ]:
df_orcamento_r = pd.concat(
    [load_orcamento_r(ano).assign(ANO=str(ano)) for ano in ANOS],
    ignore_index=True
)
df_orcamento_r

In [ ]:
df_orcamento_r.loc[:, 'VALOR_DETALHAMENTO_AÇÃO'] = df_orcamento_r['VALOR_DETALHAMENTO_AÇÃO'].str.replace(',', '.').astype(float)
df_orcamento_r

In [ ]:
filtro_orcamento_r = (
    (df_orcamento_r['CÓDIGO_FUNÇÃO']=='16') |
    (df_orcamento_r['CÓDIGO_PROGRAMA']=='3008') |
    (df_orcamento_r['CÓDIGO_PROJ_ATIV'].isin(['1702', '1703']))
)

df_orcamento_r = df_orcamento_r.loc[filtro_orcamento_r]
df_orcamento_r = df_orcamento_r.loc[df_orcamento_r['ANO_LIQUIDAÇÃO']==df_orcamento_r['ANO_EMPENHO']]
df_orcamento_r

Vamos conferir qual o percentual do orçamento está presente na tabela de detalhamento e regionalizado a nível de subprefeitura.

In [ ]:
df_orcamento_r['VALOR_DETALHAMENTO_AÇÃO'].sum()/df_orcamento['Vl_Liquidado'].sum()

95,8% de detalhamento é um percentual excelente. Vamos checar a regionalização.

In [ ]:
(
    df_orcamento_r
    .assign(regionalizado=df_orcamento_r['SUBPREFEITURA'].str.contains('Supra')==False)
    .groupby('regionalizado')
    ['VALOR_DETALHAMENTO_AÇÃO'].sum()/df_orcamento['Vl_Liquidado'].sum()
)

O percentual de regionalização é de 80,8%. Menor, mas ainda parece ser satisfatório. Vamos manter as informações de regionalização também.

# Exportando os arquivos

Neste notebook, vamos apenas salvar os arquivos extraídos na pasta de entrada de dados.

In [ ]:
output_dir = path.join('data', 'cache', 'urbanismo')

if not path.exists(output_dir):
    makedirs(output_dir)

for c, df in [('orcamento_urbanismo_original', df_orcamento),
               ('orcamento_regionalizado_urbanismo_original', df_orcamento_r),
               ]:
    filename=path.join(output_dir, c)
    df.to_csv(f'{filename}.csv',
              sep=';',
              decimal=',',
              encoding='utf8',
              index=False
              )